In [2]:
import numpy as np
import glob
import lgdo.lh5 as lh5
import os, json
import copy
import glob
import matplotlib.pyplot as plt
from pygama.pargen.utils import load_data
from legendmeta import LegendMetadata
from dbetto import Props, TextDB, AttrsDict
import pandas as pd
from tqdm.notebook import tqdm
import awkward as ak
from resolution_extraction import get_eres_per_detector, get_expo_per_detector


from helper_lib import *

%matplotlib inline
%load_ext autoreload
%autoreload 2

In [3]:
version = "v2.1.5"
base = f"/global/cfs/projectdirs/m2676/data/lngs/l200/public/prodenv/prod-blind/ref/{version}/"
scratch_folder = "/pscratch/sd/b/borrfran/sim-v1.1.0-20260401/"
# metadata2_path = os.environ["METADATA"]
metadata2_path = "/global/homes/b/borrfran/workspace/l200/legend-metadata"
config = Props.read_from(base+"/config.json", subst_pathvar=True)['setups']['l200']['paths']


meta = LegendMetadata(config['metadata'])
meta2 = LegendMetadata(metadata2_path)

timestamp = meta.dataprod.runinfo.p03.r000.phy.start_key

chmap = meta.channelmap(timestamp)

DET_TYPES = ("BEGe", "COAX", "ICPC", "PPC")

simulated_energies = range(200, 1100, 100)

could not scan file /global/cfs/cdirs/m2676/users/borrfran/l200/legend-metadata/jldataprod/config/evt/p14_r0%%_evt_phy_overwrite.yaml, reason: ParserError('while parsing a block mapping', <yaml._yaml.Mark object at 0x7ff549838220>, 'did not find expected key', <yaml._yaml.Mark object at 0x7ff549838130>)
could not scan file /global/cfs/cdirs/m2676/users/borrfran/l200/legend-metadata/jldataprod/config/evt/p14_r0%%_evt_phy_overwrite.yaml, reason: ParserError('while parsing a block mapping', <yaml._yaml.Mark object at 0x7ff5498389a0>, 'did not find expected key', <yaml._yaml.Mark object at 0x7ff549838860>)
could not scan file /global/cfs/cdirs/m2676/users/borrfran/l200/legend-metadata/jldataprod/config/evt/p14_r0%%_evt_phy_overwrite.yaml, reason: ParserError('while parsing a block mapping', <yaml._yaml.Mark object at 0x7ff54985c0e0>, 'did not find expected key', <yaml._yaml.Mark object at 0x7ff54985e020>)
could not scan file /global/cfs/cdirs/m2676/users/borrfran/l200/legend-metadata/jldat

In [4]:
periods_simulated = ['p03', 'p04', 'p06', 'p07', 'p08', 'p09']
periods_metadata = meta2.datasets.runlists['valid']['phy']

periods_dict = {
    k: [item for rng in v for item in expand_range(rng)]
    for k, v in periods_metadata.items()
}

keys = list(periods_dict.keys())
for key in keys:
    if key not in periods_simulated:
        periods_dict.pop(key)

# periods_dict['p09'].remove('r004')

In [5]:
periods_dict

{'p03': ['r000', 'r001', 'r002', 'r003', 'r004', 'r005'],
 'p04': ['r000', 'r001', 'r002', 'r003'],
 'p06': ['r000', 'r001', 'r002', 'r003', 'r004', 'r005'],
 'p07': ['r002', 'r003', 'r004', 'r005', 'r006', 'r007'],
 'p08': ['r000',
  'r001',
  'r002',
  'r003',
  'r004',
  'r006',
  'r007',
  'r008',
  'r009',
  'r010',
  'r011',
  'r012',
  'r013',
  'r014'],
 'p09': ['r000', 'r001', 'r002', 'r003', 'r004', 'r005']}

## Populate 'expo_dict'

In [33]:
expo_dict = get_expo_per_detector(meta2, periods_dict)
Props.write_to("./v1/dictionaries/expo_dict_withp09-r004.yaml", expo_dict)

p09-r005 (expo): 100%|██████████| 101/101 [00:00<00:00, 4782.13it/s]


## Read 'expo_dict' from file

In [34]:
DET_TYPE_MAP = {"B": "BEGe", "C": "COAX", "V": "ICPC", "P": "PPC"}

expo_dict = Props.read_from("./v1/dictionaries/expo_dict_withp09-r004.yaml")

In [36]:
exposures = {
    'ICPC': [],
    'BEGe': [],
    'PPC': [],
    'COAX': []
}

for p in periods_dict:
    for r in periods_dict[p]:

        for key, values in expo_dict[p][r].items():

            if expo_dict[p][r][key]['usability'] != 'on': continue
            exposures[DET_TYPE_MAP[key[0]]].append(expo_dict[p][r][key]['expo'])
        

In [37]:
exposures_sum = {}
for key in exposures:
    exposures_sum[key] = np.sum(exposures[key])

In [38]:
exposures_sum

{'ICPC': 45.89325228610541,
 'BEGe': 10.911190302177605,
 'PPC': 9.661264611630795,
 'COAX': 7.759765955585976}